# 07 — H1 and H3 at the layer-10 operating point

**The main line.** `docs/HANDOVER.md` §"Where to go next" item 3: re-run the AdaSS hypotheses in a
regime where the model stays coherent. Every earlier test of them ran at layer 16, where dense
steering at the operating point breaks 37.5% of generations — so H1 and H3 were being compared
between configurations that had already destroyed the output, and ranked by a graded metric that
only reads the opening of a reply.

Three things changed underneath those tests, and each one changes how they must be run:

| what changed | consequence for H1/H3 |
|---|---|
| The operating point moved to **layer 10, relative strength 1.0** (nb 05 §6–§7) | run there, not at layer 16 |
| Strength is **`‖m·v‖ / ‖h‖`**, not the raw multiplier | schemes must be compared at matched *relative* strength — matched raw multiplier across masking schemes is what made week 2's H1 test unfair, and `apply_scaling` only half-fixes it |
| Coherence and answering are **measurable per generation** (nb 04, hand-confirmed at layer 10 in nb 06) | rank by behaviour, not by `refusal_margin` — which peaks before the model breaks and cannot see the collapse |

**The problem this notebook has to solve first.** At the new operating point dense steering scores
**100% clean refusal at 0% broken**. The AdaSS thesis is that sparsifying the intervention *recovers
output quality*; at layer 10 rel 1.0 there is no quality to recover, because the damage axis is
already on the floor. A comparison run there can only produce ties at the ceiling. So §2 first finds
where layer 10 *does* break, and the hypotheses are then tested at two points: the operating point
(where KL is the only damage axis with any range left) and the damage regime above it (where
coherence has range). Both are pre-registered below, including the outcome where the answer is
"there was nothing to recover" — which is itself a result about the AdaSS premise.

**Cost.** ~35 generation cells at n=48 × 128 tokens, plus teacher-forced KL. About 1.5 hours on a T4.
`ADASS_CONFIRM=1` adds an n=96 confirmation of the decided winners (~15 minutes).

**Order.** §0 → §1 (instruments + replication gate) → §2 (calibration, and it sets `REL_STAR`, which
§4–§6 depend on) → §3 (masks) → §4 (H1) → §5 (H2) → §6 (H3) → §7 (confirmation) → §8 (summary).

## §0 Setup

In [ ]:
# %% 0.0 BOOTSTRAP -- run this first, always. Identical locally and on Colab.
import os, subprocess, sys
from pathlib import Path

GITHUB_REPO = "YarinShitrit/adass"                 # from `git remote -v`
DRIVE_DIR   = "/content/drive/MyDrive/adass"       # fallback if you skip GitHub

IN_COLAB = "google.colab" in sys.modules


def _find_root(start):
    """Walk up looking for the repo: a pyproject.toml sitting next to the adass package."""
    p = Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "adass" / "core.py").is_file():
            return cand
    return None


def _early_env():
    """Read a .env BEFORE the package exists. Mirrors adass.env, which is the canonical copy.

    Duplicated here because on Colab this cell runs before the repo is cloned and before anything
    is pip-installed -- and the GitHub token needed to perform the clone has to come from
    somewhere. Which is also why the repo's own .env cannot be that somewhere: .env is gitignored,
    so a clone never contains one. Keep a filled-in .env on Drive; it survives runtimes.
    """
    for p in (Path.cwd() / ".env", Path("/content/drive/MyDrive/adass/.env"),
              Path("/content/drive/MyDrive/.env"), Path("/content/.env"), Path.home() / ".env"):
        if p.is_file():
            for line in p.read_text(encoding="utf-8").splitlines():
                line = line.strip().removeprefix("export ")
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, _, v = line.partition("=")
                v = v.strip().strip("\"'")
                if v and not os.environ.get(k.strip()):
                    os.environ[k.strip()] = v
            print(f"loaded .env from {p}")
            return p
    return None


def _secret(name, prompt):
    """Environment (incl. .env) -> Colab Secrets -> prompt. Nothing is stored in the notebook."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # browser Colab frontend only
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            return v
    except Exception:
        pass
    import getpass
    v = getpass.getpass(prompt)
    if v:
        os.environ[name] = v
    return v


_early_env()
ROOT = _find_root(Path.cwd())

if IN_COLAB and ROOT is None:
    if not os.environ.get("GH_TOKEN") and Path("/content/drive").exists() is False:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            _early_env()
        except Exception:
            pass
    token = _secret("GH_TOKEN", "GitHub PAT (read access to the repo): ")
    if token:
        subprocess.run(["git", "clone", "--quiet",
                        f"https://{token}@github.com/{GITHUB_REPO}.git", "/content/adass"],
                       check=True)
        ROOT = Path("/content/adass")
    else:
        ROOT = _find_root(DRIVE_DIR) or Path(DRIVE_DIR)

assert ROOT is not None, "repo not found -- set GITHUB_REPO, or put the repo at DRIVE_DIR"
os.chdir(ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
elif str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import adass
adass.load_env()
# Both sources are GATED: HF_TOKEN needs google/gemma-2-2b-it AND walledai/AdvBench accepted.
# NOTE: HF_HUB_OFFLINE is deliberately NOT set -- nothing is cached on a fresh runtime.
from huggingface_hub import get_token
if not get_token():
    adass.require("HF_TOKEN", "HuggingFace token (gemma-2-2b-it + AdvBench accepted): ")

import torch
print(adass.paths.describe())
print(adass.env.status())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none -- everything past §1 will be very slow")

In [ ]:
# %% 0.1 Run flags, the truncation point, and the prior run.
import json, math, itertools
from collections import Counter

OUT = "week5_h1h3.json"            # adass.save_results resolves bare names to data/results/

LOAD_MODEL = os.environ.get("ADASS_LOAD_MODEL", "0") == "1"
CONFIRM    = os.environ.get("ADASS_CONFIRM", "0") == "1"       # §7, n=96, only after §4-§6 decide
print(f"LOAD_MODEL={LOAD_MODEL}  CONFIRM={CONFIRM}   "
      "(set ADASS_LOAD_MODEL=1 for everything past §1)")
RESULTS = {}

# save_results MERGES at the top level, so no later cell can delete a section it did not compute.
# §0.1 is the one deliberate truncation point, and it fires only on a full run: a CPU-only pass
# cannot regenerate the GPU sections, so rotating them away would destroy the only copy. That is
# not hypothetical -- it happened on 21 August, and again in a milder form on 25 August, which is
# why the `and LOAD_MODEL` is there.
_out = adass.results_path(OUT)
_prev = _out.with_suffix(".prev.json")
if _out.exists() and LOAD_MODEL:
    _out.replace(_prev)
    print(f"rotated {_out.name} -> {_prev.name}  (full run: starting clean)")
elif _out.exists():
    print(f"CPU-only run: MERGING into existing {_out.name}, not rotating.")

# The verdict cells re-run on CPU against whatever the last GPU run left behind, so every one of
# them reads through `stored()` rather than off a local variable that only exists mid-run.
PRIOR = {}
for _p in (_out, _prev):
    if _p.exists():
        PRIOR = json.load(open(_p))
        print(f"prior run loaded from {_p.name}: {list(PRIOR)}")
        break


def stored(key, default=None):
    """This run's value if this run computed it, else the previous run's."""
    return RESULTS.get(key, PRIOR.get(key, default))

In [ ]:
# %% 0.2 Module provenance. The stored judge output is only reusable if the prompts still hash
# to the version it was produced under -- see HANDOVER trap 2 for what silent drift cost here.
print("adass      ", adass.__version__, "from", Path(adass.__file__).parent)
print("judge hash ", adass.judge_prompt_hash())
_stored_hash = json.load(open(adass.artifact("steps123_results.json")))["step3"]["prompt_hash"]
print("stored hash", _stored_hash,
      "-- MATCH" if adass.judge_prompt_hash() == _stored_hash else "-- CHANGED: do not reload")
assert adass.judge_prompt_hash() == _stored_hash, (
    "judge prompts changed: every comparison in this notebook against a stored week-4 number "
    "would be measuring two different instruments. Bump the version deliberately or revert.")

In [ ]:
# %% 0.3 Environment, splits, vectors. float16 is a STOP condition: Gemma-2 emits broken text in
# fp16, and that failure is visually identical to the degeneration this project studies.
import torch, transformers

DEV, DT = adass.pick_device(), adass.pick_dtype(adass.pick_device())
print("device", DEV, "| dtype", DT)
assert DT is not torch.float16, "float16: STOP. See README, environment check."

CONFIG = json.load(open(adass.paths.config()))
LAYER  = CONFIG["best_layer"]              # 10, set from evidence on 24 August
R16    = CONFIG["r16"]                     # 0.5537 -- relative strength of the raw vector at L16
assert LAYER == 10, f"config says layer {LAYER}; this notebook is written for the layer-10 point"

SPL     = adass.make_splits(seed=CONFIG["seed"])   # train_n MUST stay at its default 128 --
PROMPTS = SPL["harmless_test"]                     # train_n=160 shifts harmless_test by 32 items
assert len(PROMPTS) == 48
DIRS = torch.load(adass.artifact("refusal_dirs.pt"))
V    = DIRS[LAYER + 1]                             # refusal_dirs is indexed [layer + 1]
print(f"{len(PROMPTS)} test prompts | layer {LAYER} | ||V|| {float(V.norm()):.3f} "
      f"| r16 {R16:.4f}")

MAXNEW  = 128          # 48 tokens cannot show apology-then-answer; week 3 §2 is why this is 128
GEN_BS  = 8
KL_BS   = 4            # teacher-forced logits are [B, T, 256k]; 4 keeps a T4 inside its memory
KL_REF_TOKENS = 48     # the fixed reference text, as in week 3 §5.1
KL_WINDOW = 8          # see §5: the shared window that makes position schemes comparable

RESULTS["env"] = dict(load_model=LOAD_MODEL, device=DEV, dtype=str(DT), torch=torch.__version__,
                      transformers=transformers.__version__, layer=LAYER, r16=R16,
                      max_new_tokens=MAXNEW, n_prompts=len(PROMPTS),
                      gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
                      bf16_supported=(torch.cuda.is_bf16_supported()
                                      if torch.cuda.is_available() else None))
print("env:", RESULTS["env"])
print(adass.save_results(RESULTS, OUT))

MODEL = TOK = TO_CHAT = None
if LOAD_MODEL:
    MODEL, TOK, DT, DEV = adass.load_model()
    TO_CHAT = adass.make_chat_fn(TOK)
    print("model loaded")

## §1 The instruments, and the gate they have to pass

Same two axes as notebook 05, and for the same reasons: **coherence from the mechanical detector**,
which passes both of its controls unconditionally, and **answering from the binary judge**, which is
91.4% against the 160 hand labels. The judge's *coherence* question is not used — it fails its
positive control at 72.9% against a bar of 95%.

`clean_refusal = coherent AND not answered` is the top-right corner of the plan's frontier and the
quantity every verdict below is written against.

At layer 10 both instruments were validated out-of-distribution until notebook 06 closed that gap:
42 blind labels at this operating point, 100% agreement on both axes over stratum A (n=29,
Wilson lower bound 0.88). That is what licenses using them here without a fresh labelling pass.

**§1.2 is a blocking gate.** It reproduces `L10/rel1.0` from `week4_layers.json` §7 with the code
path this notebook uses — a *different* code path, because §3 replaces the multiplier with an
exactly norm-matched vector. If the two disagree, the normalisation is not doing what it claims and
nothing downstream is comparable to the week-4 numbers.

In [ ]:
# %% 1.1 Fit the mechanical thresholds on the anchors, and define the two axes.
GENS = json.load(open(adass.artifact("week3_generations.json")))
FIT = adass.fit_coherence_thresholds(GENS["no-steer"], GENS["dense/all m=2"])
for feat, d in FIT.items():
    print(f"  {feat:12} thr={d['threshold']:8.3f}  bacc={d['balanced_acc']:.3f}  margin={d['margin']:+.3f}")


def mech_broken(texts):
    return [adass.classify_mechanical(t, FIT)["broken"] for t in texts]


def judge_answered(prompts, texts):
    """None when the model is not loaded, so every section still completes."""
    if not LOAD_MODEL:
        return None
    return [o["answered"] for o in
            adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(prompts, texts)))]


def score(texts, prompts, answered=None):
    """The row every table in this notebook reports. Wilson CIs on all three rates.

    CIs are stored as [lo, hi] -- wilson_ci returns (point, lo, hi) and the point estimate is
    already the neighbouring field. Indexing rather than unpacking that tuple is what cost a
    blocking control its meaning on 23 August (WORKLOG correction 14), so the slice is explicit.
    """
    br = mech_broken(texts)
    ans = judge_answered(prompts, texts) if answered is None else answered
    n = len(texts)
    row = dict(n=n, broken=sum(br) / n, broken_ci=list(adass.wilson_ci(sum(br), n))[1:],
               matcher=adass.refusal_rate(texts))
    if ans is not None:
        clean = [(not b) and (not a) for b, a in zip(br, ans)]
        row.update(suppressed=1 - sum(ans) / n,
                   suppressed_ci=list(adass.wilson_ci(n - sum(ans), n))[1:],
                   clean_refusal=sum(clean) / n,
                   clean_refusal_ci=list(adass.wilson_ci(sum(clean), n))[1:],
                   judge_answered=ans)
    row["mech_broken"] = br
    return row


def fmt(row, label=""):
    s = f"{label:30} broken {row['broken']:6.1%}"
    if "clean_refusal" in row:
        s += f" | suppressed {row['suppressed']:6.1%} | CLEAN {row['clean_refusal']:6.1%}"
    if "kl" in row:
        s += f" | KL {row['kl']:6.3f}"
    return s + f" | matcher {row['matcher']:6.1%}"


def disjoint(a, b):
    """Do two [lo, hi] intervals fail to overlap? The only evidence a cell is DECIDED at n=48."""
    return a[1] < b[0] or b[1] < a[0]

In [ ]:
# %% 1.2 The replication gate, plus the negative control and the KL reference text.
#
# Three things at once, all from the unsteered and the dense rel-1.0 conditions:
#   - REF_TEXTS  -- the fixed unsteered continuation every KL in this notebook is measured on;
#   - the negative control -- unsteered must be ~0% suppressed and ~0% broken, or the
#     instruments are wrong before any comparison starts;
#   - the gate -- dense at rel 1.0 must land where week 4 §7 left it.
if LOAD_MODEL:
    HN10 = adass.mean_hidden_norm(MODEL, TOK, TO_CHAT, PROMPTS, LAYER, device=DEV)
    print(f"mean ||h|| at layer {LAYER}: {HN10:.1f}  (week 4 measured 170.9)")

    REF_TEXTS = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0,
                               max_new_tokens=KL_REF_TOKENS, batch_size=GEN_BS,
                               device=DEV, dtype=DT)
    ns_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0, max_new_tokens=MAXNEW,
                             batch_size=GEN_BS, device=DEV, dtype=DT)
    ns = score(ns_gens, PROMPTS)
    print(fmt(ns, "no-steer (negative control)"))

    V_ref = adass.rel_norm_rows(V, R16 * 1.0, HN10)      # the operating point, norm-matched
    d10_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=LAYER, vector=V_ref, mult=1.0,
                              positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                              device=DEV, dtype=DT)
    d10 = score(d10_gens, PROMPTS)
    d10.update(adass.strength_row(V_ref, 1.0, HN10), label="dense", rel_factor=1.0,
               method="dense", sparsity=0.0, positions="all", n_prompts=len(PROMPTS))
    print(fmt(d10, "dense/all rel 1.0"))

    w4 = json.load(open(adass.artifact("week4_layers.json")))["s7_relative_grid"]["cells"]["L10/rel1.0"]["row"]
    print(f"\nweek 4 §7 L10/rel1.0: broken {w4['broken']:.1%}  clean {w4['clean_refusal']:.1%}")

    # The bar is 10 points, not equality: the 24 August run reproduced every rate in the sweep to
    # within 4.2% while only 14-48% of individual generations matched token-for-token, because a
    # T4 has no bfloat16 and greedy decoding is deterministic given identical numerics and not
    # otherwise. Rates are dtype-invariant here; text is not.
    gate = dict(
        neg_control_ok=bool(ns["broken"] <= 0.05 and ns["suppressed"] <= 0.05),
        clean_delta=d10["clean_refusal"] - w4["clean_refusal"],
        broken_delta=d10["broken"] - w4["broken"],
        h_norm=HN10, h_norm_w4=CONFIG["h_norms"][str(LAYER)])
    gate["repro_ok"] = bool(abs(gate["clean_delta"]) <= 0.10 and abs(gate["broken_delta"]) <= 0.10)
    gate["pass_"] = bool(gate["neg_control_ok"] and gate["repro_ok"])
    print(f"\nnegative control {'PASS' if gate['neg_control_ok'] else 'FAIL'} | "
          f"replication delta clean {gate['clean_delta']:+.1%} broken {gate['broken_delta']:+.1%} "
          f"-> {'PASS' if gate['repro_ok'] else 'FAIL'}")
    RESULTS["s1_gate"] = dict(gate=gate, no_steer=ns, dense_rel1=d10)
    RESULTS["s1_ref_texts"] = REF_TEXTS
    print(adass.save_results(RESULTS, OUT))
    assert gate["pass_"], "BLOCKING: fix this before running anything below."
else:
    HN10 = CONFIG["h_norms"][str(LAYER)]
    REF_TEXTS = (PRIOR.get("s1_ref_texts") or None)
    ns = d10 = None
    print(f"deferred: needs ADASS_LOAD_MODEL=1. Using stored ||h|| = {HN10}")

## §2 Calibration — where does layer 10 actually break?

**Why this section exists.** AdaSS's premise is that dense steering damages output and that
sparsifying it recovers quality. At the operating point that premise is not satisfied: dense/all at
rel 1.0 is 100% clean refusal at **0% broken**, and week 4 §7 shows it still holds 97.9% / 0% at
1.5×. Comparing masking schemes there can only produce ties against a ceiling, and a tie against a
ceiling is not evidence about H1.

So: push dense/all up the relative axis until it breaks, and run the hypotheses at both ends —
the operating point, where **KL** is the damage axis with range left, and the damage regime, where
**coherence** has range.

**Pre-registered, before the run.**

> `REL_STAR` is the **smallest** factor in `{1.5, 2.0, 2.5, 3.0, 4.0}` at which dense/all reaches
> `broken ≥ 0.25`. If no factor does, `REL_STAR` is `None` and the reading is recorded as
> **"no damage regime below 4×"** — layer 10 tolerates the whole ladder, H1/H3 are tested at the
> operating point on KL alone, and the AdaSS premise is reported as *unsatisfied at a correctly
> selected operating point*.

That last outcome is not a failure of the run. "The intervention this method was designed to repair
does not need repairing once the layer is chosen on evidence" is a claim about the method's
motivation, and it follows from the same measurement error that runs through the whole project.

In [ ]:
# %% 2.1 The damage-onset ladder. Dense, all positions, layer 10, climbing the relative axis.
LADDER = [1.5, 2.0, 2.5, 3.0, 4.0]
BREAK_BAR = 0.25

if LOAD_MODEL:
    onset = {"rel1.0": dict(row=d10, gens=d10_gens)}     # already run as the §1.2 gate
    for f in LADDER:
        vec = adass.rel_norm_rows(V, R16 * f, HN10)
        g = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=LAYER, vector=vec, mult=1.0,
                           positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                           device=DEV, dtype=DT)
        r = score(g, PROMPTS)
        r.update(adass.strength_row(vec, 1.0, HN10), rel_factor=f)
        onset[f"rel{f}"] = dict(row=r, gens=g)
        print(fmt(r, f"dense/all rel x{f}"))
        adass.empty_cache(DEV)

    broke = [f for f in LADDER if onset[f"rel{f}"]["row"]["broken"] >= BREAK_BAR]
    REL_STAR = min(broke) if broke else None
    RESULTS["s2_damage_onset"] = dict(cells=onset, ladder=LADDER, break_bar=BREAK_BAR,
                                      rel_star=REL_STAR)
    print(adass.save_results(RESULTS, OUT))
else:
    REL_STAR = (stored("s2_damage_onset") or {}).get("rel_star")

print(f"\nREL_STAR = {REL_STAR}"
      + ("  -- the damage regime; H1/H3 run here AND at rel 1.0" if REL_STAR else
         "  -- NO damage regime below 4x. The AdaSS premise is unsatisfied at this operating"
         " point; H1/H3 fall back to KL at rel 1.0 (see §4)."))
REL_POINTS = [1.0] + ([REL_STAR] if REL_STAR else [])
print("REL_POINTS =", REL_POINTS)

## §3 The masks, and what "matched strength" has to mean here

Four scores over the 2304 dimensions of `V`, one static and three per input:

| method | score | what it ranks by |
|---|---|---|
| `static` | `|V|` | `V`'s own largest components. One mask for every input |
| `absproj` | `|V ⊙ (h − μ)|` | which dimensions *this* input already loads on |
| `signed` | `V ⊙ (h − μ)` | the same, signed — dimensions pushing *toward* refusal |
| `grad` | `|∂ℒ/∂h ⊙ V|` | which dimensions actually change the outcome, `ℒ = refusal_margin` |

**The renormalisation confound, and why this notebook does not use `apply_scaling`.** Zeroing 90% of
a vector shortens it, and by a different amount per method and per prompt: static keeps 68.9% of
`‖V‖` *by construction* — it selects the largest components — while the adaptive scores keep
55.7–56.8%. Week 2 rescaled back to `‖V‖` and then compared at a matched **multiplier**, which
pushed adaptive masks 1.76–1.80× against static's 1.45×. Its H1 test could not have been fair
whichever method is better.

Week 3's answer was to sweep the multiplier per method and compare at matched KL. This notebook
takes the sharper one now available: `adass.rel_norm_rows` sets **every row of every scheme to the
same perturbation norm `rel · ‖h‖`**, and steers with multiplier 1.0. Under exact norm matching the
choice of scaling rule is moot — `"none"` and `"match_norm"` produce the same vector — so the
confound is dissolved rather than compensated for. §3.1 prints the retained-norm fractions anyway,
because that diagnostic is what the confound *was*.

One thing exact matching does **not** buy: it equalises the size of the perturbation, not its
direction. A sparse vector at the same norm points somewhere else in the residual stream, and that
is the intervention being tested.

In [ ]:
# %% 3.1 Per-prompt inputs, the four mask builders, and the retained-norm diagnostic.
SPARSITIES = [0.90, 0.99]

if LOAD_MODEL:
    MU = adass.last_token_hidden(MODEL, TOK, TO_CHAT, SPL["harmless_train"],
                                 device=DEV).mean(1)[LAYER + 1]         # [d]
    H_TEST = adass.last_token_hidden(MODEL, TOK, TO_CHAT, PROMPTS, device=DEV)[LAYER + 1]
    GRAD = adass.grad_scores(MODEL, TOK, TO_CHAT, PROMPTS, V, LAYER,
                             device=DEV, dtype=DT, batch_size=4)
    print("mu", tuple(MU.shape), "| h_test", tuple(H_TEST.shape), "| grad", tuple(GRAD.shape))

    def build_vectors(method, sparsity):
        """-> [N, d] per-prompt MASKED vectors, unscaled. rel_norm_rows does the scaling."""
        n = len(PROMPTS)
        if method == "static":
            return (V * adass.static_mask(V, sparsity)).unsqueeze(0).expand(n, -1).contiguous()
        out = torch.empty(n, V.numel())
        for i in range(n):
            if method == "absproj":
                m = adass.adaptive_absproj_mask(V, H_TEST[i], MU, sparsity)
            elif method == "signed":
                m = adass.adaptive_signed_mask(V, H_TEST[i], MU, sparsity)
            elif method == "grad":
                m = adass.topk_mask(GRAD[i], sparsity)
            else:
                raise ValueError(method)
            out[i] = V * m
        return out

    print(f"\n{'method':10s} {'sparsity':>9s} {'kept dims':>10s} {'retained ||v||':>15s} "
          f"{'week-2 alpha':>13s}")
    diag = []
    for method in ["static", "absproj", "signed", "grad"]:
        for s in SPARSITIES:
            vecs = build_vectors(method, s)
            frac = float(vecs.norm(dim=-1).mean()) / float(V.norm())
            kept = int((vecs[0] != 0).sum())
            diag.append(dict(method=method, sparsity=s, kept=kept, retained_frac=frac,
                             match_norm_alpha=1 / frac))
            print(f"{method:10s} {s:9.2f} {kept:10d} {frac:15.3f} {1/frac:13.2f}")
    RESULTS["s3_scaling_diagnostic"] = diag
    print("\nThose alphas are week 2's H1 test: at a matched multiplier the adaptive schemes were"
          "\nsteered that many times harder than static, per prompt. Exact norm matching below"
          " removes the axis they differ on.")
    print(adass.save_results(RESULTS, OUT))


def run_config(label, base_vec, rel_factor, positions="all", prompts=None, extra=None):
    """Generate + score one configuration at an exactly matched relative strength."""
    prompts = PROMPTS if prompts is None else prompts
    rel = R16 * rel_factor
    vec = adass.rel_norm_rows(base_vec, rel, HN10)
    g = adass.generate(MODEL, TOK, TO_CHAT, prompts, layer=LAYER, vector=vec, mult=1.0,
                       positions=positions, max_new_tokens=MAXNEW, batch_size=GEN_BS,
                       device=DEV, dtype=DT)
    row = score(g, prompts)
    row.update(adass.strength_row(vec, 1.0, HN10), label=label, rel_factor=rel_factor,
               rel_target=rel, positions=str(positions), n_prompts=len(prompts))
    adass.empty_cache(DEV)
    if extra:
        row.update(extra)
    return row, g, vec


def add_kl(row, vec, positions="all", window=None):
    """KL(steered || base) on the fixed reference, bootstrap CI. Mutates and returns `row`."""
    kls = adass.kl_vs_base(MODEL, TOK, TO_CHAT, PROMPTS, REF_TEXTS, LAYER, vec, 1.0,
                           positions, device=DEV, dtype=DT, batch_size=KL_BS,
                           per_prompt=True, window=window)
    m, lo, hi = adass.bootstrap_ci(kls)
    key = "kl" if window is None else f"kl_w{window}"
    row[key] = m
    row[key + "_ci"] = [lo, hi]
    adass.empty_cache(DEV)
    return row

## §4 H1 — per-input masks against a static mask, pre-registered

> **H1** (original wording): *per-input sparse masks match dense steering at ≥90% sparsity and beat
> an equal-sparsity static mask on held-out prompts.*

Two claims, tested separately, both at matched relative strength and matched sparsity.

**H1a — retention.** A scheme *retains* if its suppression CI overlaps dense/all's at the same
relative strength. This is "match dense steering", and it is a CI-overlap test, so a scheme that
merely fails to differ counts as retaining — deliberately, since that is what the hypothesis claims.

**H1b — adaptive beats static.** Decided cells only. The axis depends on which regime the cell is
in, because the two regimes have range in different places:

| regime | primary axis | precondition |
|---|---|---|
| rel 1.0, both schemes at `broken ≤ 5%` | **KL**, lower wins | suppression CIs must overlap — otherwise the KL comparison is between different effect sizes |
| `REL_STAR`, damage present | **clean refusal**, higher wins | none |

A cell is **decided** when the two CIs on the primary axis are disjoint. n=48 gives roughly ±14
points on a rate, so most cells will not decide, and that is reported rather than papered over.

**Verdict rule, fixed now:**

| outcome | reading |
|---|---|
| adaptive wins ≥1 decided cell and loses none | **H1 supported** |
| static wins ≥1 decided cell and adaptive wins none | **H1 rejected** |
| both win decided cells | **H1 mixed** — report per cell, no headline |
| no decided cells | **H1 undecided at n=48** — escalate the closest pair to n=96 in §7 |

**And the outcome that is about the premise rather than the hypothesis.** If dense/all at rel 1.0
has `broken ≤ 5%` *and* no scheme beats it on KL with disjoint CIs, record
**"nothing to recover"**: at an operating point selected on evidence, dense steering already costs
no measurable coherence, so the motivation for sparsifying it does not hold here. That is a result
about AdaSS's premise, and it is pre-registered so that it cannot be read as a disappointing null.

In [ ]:
# %% 4.1 The H1 grid. Every cell at exactly matched relative strength.
MASK_SPECS = [("dense", None, 0.00)] + [
    (f"{m}-{s:.2f}", m, s) for s in SPARSITIES
    for m in ["static", "absproj", "signed", "grad"]]
ADAPTIVE = ("absproj", "signed", "grad")   # a tuple, not a set: the tables below print in
#                                            this order, and a set would reshuffle them per run

if LOAD_MODEL:
    h1_cells = {}
    for f in REL_POINTS:
        for label, method, sp in MASK_SPECS:
            base = V if method is None else build_vectors(method, sp)
            row, gens, vec = run_config(label, base, f,
                                        extra=dict(method=method or "dense", sparsity=sp))
            add_kl(row, vec, "all")
            add_kl(row, vec, "all", window=KL_WINDOW)   # §6 compares gated against ungated
            h1_cells[f"{label}/rel{f}"] = dict(row=row, gens=gens)
            print(fmt(row, f"{label:14s} rel x{f}"))
    # dense at rel 1.0 is generated twice -- once as the §1.2 gate, once here as a grid cell --
    # and greedy decoding is deterministic, so the two MUST agree. Cheap check, and it catches a
    # stray nondeterminism (a moved split, a changed dtype mid-session) before the verdicts read
    # the grid.
    _gate = (RESULTS.get("s1_gate") or {}).get("dense_rel1")
    if _gate:
        _d = h1_cells["dense/rel1.0"]["row"]
        print()
        print(f"dense/rel1.0 here {_d['clean_refusal']:.1%} clean vs the §1.2 gate row "
              f"{_gate['clean_refusal']:.1%} -- "
              + ("identical" if _d["clean_refusal"] == _gate["clean_refusal"]
                 else "DIFFERENT: something moved between §1 and §4"))
    RESULTS["s4_h1"] = dict(cells=h1_cells, rel_points=REL_POINTS,
                            specs=[[a, b, c] for a, b, c in MASK_SPECS])
    print(adass.save_results(RESULTS, OUT))
else:
    h1_cells = (stored("s4_h1") or {}).get("cells")
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 4.2 Apply the pre-registered rule. Printed as a verdict, not as a paragraph to interpret later.
h1 = stored("s4_h1")
if h1:
    cells = {k: v["row"] for k, v in h1["cells"].items()}
    rel_points = h1["rel_points"]

    def cell(label, f):
        return cells.get(f"{label}/rel{f}")

    # -- H1a: does each scheme match dense? -----------------------------------------------
    print("H1a  RETENTION -- suppression CI overlaps dense at the same relative strength\n")
    retention = []
    for f in rel_points:
        dense = cell("dense", f)
        for label, method, sp in [(a, b, c) for a, b, c in h1["specs"] if b]:
            r = cell(label, f)
            if not r:
                continue
            ok = not disjoint(r["suppressed_ci"], dense["suppressed_ci"])
            retention.append(dict(rel=f, label=label, retains=bool(ok),
                                  suppressed=r["suppressed"], dense=dense["suppressed"]))
            print(f"  rel x{f}  {label:14s} suppressed {r['suppressed']:6.1%} "
                  f"vs dense {dense['suppressed']:6.1%}   {'retains' if ok else 'DIFFERS'}")

    # -- H1b: adaptive vs static, same sparsity, same strength ----------------------------
    print("\nH1b  ADAPTIVE vs STATIC -- decided cells only\n")
    comps = []
    for f in rel_points:
        for sp in SPARSITIES:
            st = cell(f"static-{sp:.2f}", f)
            if not st:
                continue
            for m in ADAPTIVE:
                ad = cell(f"{m}-{sp:.2f}", f)
                if not ad:
                    continue
                floored = max(ad["broken"], st["broken"]) <= 0.05
                axis = "kl" if floored else "clean_refusal"
                matched = not disjoint(ad["suppressed_ci"], st["suppressed_ci"])
                if axis == "kl":
                    dec = matched and disjoint(ad["kl_ci"], st["kl_ci"])
                    win = (m if ad["kl"] < st["kl"] else "static") if dec else None
                    detail = f"KL {ad['kl']:.3f} vs {st['kl']:.3f}"
                else:
                    dec = disjoint(ad["clean_refusal_ci"], st["clean_refusal_ci"])
                    win = (m if ad["clean_refusal"] > st["clean_refusal"] else "static") if dec else None
                    detail = f"clean {ad['clean_refusal']:.1%} vs {st['clean_refusal']:.1%}"
                comps.append(dict(rel=f, sparsity=sp, adaptive=m, axis=axis, decided=bool(dec),
                                  winner=win, matched_effect=bool(matched), detail=detail))
                print(f"  rel x{f} s={sp:.2f}  {m:8s} vs static  [{axis:13s}] {detail:32s} "
                      + (f"-> {win.upper()}" if dec else
                         "-> undecided" + ("" if matched or axis == "clean_refusal"
                                           else " (effects differ; KL not comparable)")))

    dec = [c for c in comps if c["decided"]]
    ad_wins = [c for c in dec if c["winner"] in ADAPTIVE]
    st_wins = [c for c in dec if c["winner"] == "static"]
    verdict = ("H1 UNDECIDED at n=48" if not dec else
               "H1 SUPPORTED" if ad_wins and not st_wins else
               "H1 REJECTED" if st_wins and not ad_wins else
               "H1 MIXED -- report per cell")

    # -- the premise check ----------------------------------------------------------------
    d1 = cell("dense", 1.0)
    beats_dense_kl = [lbl for lbl, _, _ in h1["specs"] if lbl != "dense"
                      and cell(lbl, 1.0)
                      and not disjoint(cell(lbl, 1.0)["suppressed_ci"], d1["suppressed_ci"])
                      and disjoint(cell(lbl, 1.0)["kl_ci"], d1["kl_ci"])
                      and cell(lbl, 1.0)["kl"] < d1["kl"]]
    at_op = bool(d1["broken"] <= 0.05 and not beats_dense_kl)
    rel_star = (stored("s2_damage_onset") or {}).get("rel_star")

    print(f"\n{len(dec)}/{len(comps)} cells decided.   VERDICT: {verdict}")
    print(f"dense/all rel 1.0: broken {d1['broken']:.1%}, KL {d1['kl']:.3f}. "
          f"Schemes beating it on KL at matched effect: {beats_dense_kl or 'none'}")
    if at_op and rel_star is None:
        print("\nPREMISE: NOTHING TO RECOVER -- at an operating point selected on evidence, dense"
              "\nsteering costs no measurable coherence, no sparse scheme lowers KL at matched"
              "\neffect, and §2 found no damage regime below 4x. The motivation for sparsifying"
              "\nthis intervention does not hold anywhere on the ladder that was searched.")
    elif at_op:
        # The narrow form, and it MUST stay narrow: a damage regime exists at rel_star, so the
        # hypotheses are live there and the verdict above is what decides them. Printing the
        # sweeping version here would be the week-3.5 error exactly -- reading a property of one
        # operating point as a property of the method.
        print(f"\nPREMISE: nothing to recover AT THE OPERATING POINT (dense is already "
              f"{1 - d1['broken']:.0%} coherent there),"
              f"\nbut §2 found damage from rel x{rel_star}, so H1/H3 are decided in that regime,"
              " not at this one.")
    RESULTS["s4_h1_verdict"] = dict(verdict=verdict, comparisons=comps, retention=retention,
                                    decided=len(dec), adaptive_wins=len(ad_wins),
                                    static_wins=len(st_wins),
                                    beats_dense_kl=beats_dense_kl,
                                    nothing_to_recover_at_op=at_op,
                                    nothing_to_recover_anywhere=bool(at_op and rel_star is None),
                                    rel_star=rel_star)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: §4.1 has not run")

## §5 H2 — positions, re-measured on behaviour

> **H2**: *steering <30% of positions retains most behavioural effect with lower perplexity/KL
> degradation than all-position steering.*

Week 2 reported this as its headline. Week 3 overturned it by reading the generations: under a
`first-4` gate the model apologises and then answers in full, which the substring matcher scored as
a refusal. The judge measures that directly now — `suppressed` is "the user did not get an answer",
so apology-then-answer lands where it belongs.

**Two instruments, one asymmetry.** Under a `first-k` gate the steering is off for most of a
128-token continuation, so a full-length KL averages mostly base-like positions and reports the
*reversion* as low damage. That is the mechanism behind week 2's result, one metric over. This
section therefore reports KL twice: full-length, and restricted to the first **8** continuation
tokens (`window=KL_WINDOW`) — a window shared by every position scheme, so they are comparable.
The gap between the two columns is the asymmetry, measured.

**Pre-registered.** H2 is supported at a relative strength if some gate holds suppression
(CI overlapping all-positions) at strictly lower **windowed** KL with disjoint CIs. The matcher is
reported alongside purely as a check that its old headline reproduces — it is not evidence.

In [ ]:
# %% 5.1 Position gates at matched relative strength.
POS_SPECS = [("all", "all"), ("prompt-only", "prompt_only"), ("prompt-last", "prompt_last"),
             ("first-1", ("gen_first_k", 1)), ("first-4", ("gen_first_k", 4)),
             ("first-8", ("gen_first_k", 8))]

if LOAD_MODEL:
    pos_cells = {}
    for f in REL_POINTS:
        for label, spec in POS_SPECS:
            row, gens, vec = run_config(f"dense/{label}", V, f, positions=spec)
            add_kl(row, vec, spec)                      # full-length
            add_kl(row, vec, spec, window=KL_WINDOW)    # the shared window
            pos_cells[f"{label}/rel{f}"] = dict(row=row, gens=gens)
            print(f"{fmt(row, f'{label:12s} rel x{f}')} | KL_w{KL_WINDOW} "
                  f"{row[f'kl_w{KL_WINDOW}']:6.3f}")
    RESULTS["s5_positions"] = dict(cells=pos_cells, window=KL_WINDOW,
                                   specs=[[a, str(b)] for a, b in POS_SPECS],
                                   rel_points=REL_POINTS)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 5.2 H2 verdict, and the apology-then-answer check the matcher cannot make.
s5 = stored("s5_positions")
if s5:
    cells = {k: v["row"] for k, v in s5["cells"].items()}
    W = s5["window"]
    print(f"{'gate':14s} {'suppressed':>11s} {'broken':>8s} {'clean':>8s} {'matcher':>8s} "
          f"{'KL full':>9s} {'KL w' + str(W):>9s}")
    for k, r in cells.items():
        print(f"{k:14s} {r['suppressed']:11.1%} {r['broken']:8.1%} {r['clean_refusal']:8.1%} "
              f"{r['matcher']:8.1%} {r['kl']:9.3f} {r['kl_w' + str(W)]:9.3f}")

    rows, h2_wins = [], []
    for f in s5["rel_points"]:
        allpos = cells.get(f"all/rel{f}")
        for label, _ in s5["specs"]:
            if label == "all":
                continue
            r = cells.get(f"{label}/rel{f}")
            if not r:
                continue
            holds = not disjoint(r["suppressed_ci"], allpos["suppressed_ci"])
            cheaper = (disjoint(r[f"kl_w{W}_ci"], allpos[f"kl_w{W}_ci"])
                       and r[f"kl_w{W}"] < allpos[f"kl_w{W}"])
            # The gap the week-2 headline lived in: high matcher, low suppression.
            gap = r["matcher"] - r["suppressed"]
            rows.append(dict(rel=f, gate=label, holds_effect=bool(holds), cheaper=bool(cheaper),
                             matcher_minus_suppressed=gap,
                             kl_full=r["kl"], kl_win=r[f"kl_w{W}"]))
            if holds and cheaper:
                h2_wins.append(f"{label}/rel{f}")

    verdict = ("H2 SUPPORTED at " + ", ".join(h2_wins)) if h2_wins else \
              "H2 NOT SUPPORTED -- no gate holds the effect at strictly lower windowed KL"
    print(f"\nVERDICT: {verdict}")
    worst = max(rows, key=lambda r: r["matcher_minus_suppressed"], default=None)
    if worst:
        print(f"largest matcher-minus-suppression gap: {worst['gate']}/rel{worst['rel']} "
              f"{worst['matcher_minus_suppressed']:+.1%}  "
              "-- the size of the week-2 headline's error, measured directly")
    RESULTS["s5_h2_verdict"] = dict(verdict=verdict, rows=rows, wins=h2_wins)
    print(adass.save_results(RESULTS, OUT))

    print("\nRead four of these generations before believing any of it (HANDOVER trap 8):")
    for k in list(s5["cells"])[:2]:
        for i in (0, 1):
            print(f"\n--- {k}  prompt: {PROMPTS[i]}\n{s5['cells'][k]['gens'][i][:320]}")
else:
    print("deferred: §5.1 has not run")

## §6 H3 — the joint method

> **H3**: *the joint method achieves the best effect-vs-quality Pareto frontier among all
> ablations.*

Week 3 tested it for the first time and found the ordering **position-only > joint > mask-only** at
every KL budget — but every winning position-only configuration was a `first-k` gate, i.e. exactly
the configurations sitting in the blind spot §5 measures. So H3 needed re-measuring, not
re-interpreting, and this is that re-measurement: behavioural axes, matched relative strength,
windowed KL, at a layer where the model stays coherent.

The joint arm is the **best adaptive scheme from §4** at 0.90 sparsity crossed with the position
gates that held their effect in §5 (falling back to `first-4` and `first-8` if none did — a joint
method built only from winning components would be a rigged comparison).

**Pre-registered.** H3 is supported if a joint cell beats **both** its own components — the same
mask at all positions, and dense at the same gate — on the primary axis for that regime (clean
refusal where damage exists, windowed KL at matched suppression where it does not), with disjoint
CIs, in at least one cell and losing none. Ties are ties: "joint is no worse" does not support a
hypothesis that claims it is better.

In [ ]:
# %% 6.1 Joint = per-input mask x position gate.
v4 = stored("s4_h1_verdict") or {}
_wins = [c["adaptive"] for c in v4.get("comparisons", []) if c.get("decided")
         and c.get("winner") in ADAPTIVE]
BEST_MASK = Counter(_wins).most_common(1)[0][0] if _wins else "absproj"
print(f"best adaptive scheme from §4: {BEST_MASK}"
      + ("" if _wins else "  (no decided cell -- falling back to absproj, the week-2 default)"))

v5 = stored("s5_h2_verdict") or {}
JOINT_GATES = [g.split("/")[0] for g in v5.get("wins", [])] or ["first-4", "first-8"]
JOINT_GATES = [g for g in dict.fromkeys(JOINT_GATES) if g != "all"] or ["first-4", "first-8"]
SPEC_OF = dict(POS_SPECS)
print("joint gates:", JOINT_GATES)

if LOAD_MODEL:
    joint_cells = {}
    base = build_vectors(BEST_MASK, 0.90)
    for f in REL_POINTS:
        for g in JOINT_GATES:
            row, gens, vec = run_config(f"JOINT/{BEST_MASK}-0.90/{g}", base, f,
                                        positions=SPEC_OF[g])
            add_kl(row, vec, SPEC_OF[g])
            add_kl(row, vec, SPEC_OF[g], window=KL_WINDOW)
            joint_cells[f"{g}/rel{f}"] = dict(row=row, gens=gens)
            print(f"{fmt(row, f'JOINT {BEST_MASK}-0.90 {g:9s} rel x{f}')} | "
                  f"KL_w{KL_WINDOW} {row[f'kl_w{KL_WINDOW}']:6.3f}")
    RESULTS["s6_joint"] = dict(cells=joint_cells, mask=BEST_MASK, sparsity=0.90,
                               gates=JOINT_GATES, rel_points=REL_POINTS)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 6.2 H3 verdict: does the joint method beat BOTH of its own components?
s6, s5, s4 = stored("s6_joint"), stored("s5_positions"), stored("s4_h1")
if s6 and s5 and s4:
    W = s5["window"]
    jc = {k: v["row"] for k, v in s6["cells"].items()}
    pc = {k: v["row"] for k, v in s5["cells"].items()}
    mc = {k: v["row"] for k, v in s4["cells"].items()}

    def better(a, b):
        """(winner, decided, axis) for two rows. Damage regime -> clean refusal; else KL_w."""
        floored = max(a["broken"], b["broken"]) <= 0.05
        if floored:
            if disjoint(a["suppressed_ci"], b["suppressed_ci"]):
                return None, False, "kl_w (effects differ)"
            ka, kb = f"kl_w{W}", f"kl_w{W}"
            if ka in a and kb in b and disjoint(a[ka + "_ci"], b[kb + "_ci"]):
                return ("A" if a[ka] < b[kb] else "B"), True, "kl_w"
            return None, False, "kl_w"
        if disjoint(a["clean_refusal_ci"], b["clean_refusal_ci"]):
            return ("A" if a["clean_refusal"] > b["clean_refusal"] else "B"), True, "clean_refusal"
        return None, False, "clean_refusal"

    rows = []
    for f in s6["rel_points"]:
        for g in s6["gates"]:
            j = jc.get(f"{g}/rel{f}")
            mask_only = mc.get(f"{s6['mask']}-0.90/rel{f}")
            pos_only = pc.get(f"{g}/rel{f}")
            if not (j and mask_only and pos_only):
                continue
            wm, dm, axm = better(j, mask_only)
            wp, dp, axp = better(j, pos_only)
            beats_both = bool(dm and dp and wm == "A" and wp == "A")
            loses = bool((dm and wm == "B") or (dp and wp == "B"))
            rows.append(dict(rel=f, gate=g, axis=axm, beats_mask_only=bool(dm and wm == "A"),
                             beats_pos_only=bool(dp and wp == "A"),
                             beats_both=beats_both, loses=loses,
                             joint_clean=j["clean_refusal"], mask_clean=mask_only["clean_refusal"],
                             pos_clean=pos_only["clean_refusal"],
                             joint_klw=j.get(f"kl_w{W}"), mask_klw=mask_only.get(f"kl_w{W}"),
                             pos_klw=pos_only.get(f"kl_w{W}")))
            print(f"rel x{f} {g:9s} [{axm:22s}] joint clean {j['clean_refusal']:6.1%} | "
                  f"mask-only {mask_only['clean_refusal']:6.1%} | pos-only {pos_only['clean_refusal']:6.1%}"
                  f"  -> {'BEATS BOTH' if beats_both else ('LOSES' if loses else 'tie/undecided')}")

    won = [r for r in rows if r["beats_both"]]
    lost = [r for r in rows if r["loses"]]
    verdict = ("H3 SUPPORTED" if won and not lost else
               "H3 REJECTED" if lost and not won else
               "H3 MIXED" if won and lost else
               "H3 NOT SUPPORTED -- no decided cell where joint beats both components")
    print(f"\nVERDICT: {verdict}   ({len(won)} cells beat both, {len(lost)} lost)")
    RESULTS["s6_h3_verdict"] = dict(verdict=verdict, rows=rows, mask=s6["mask"])
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: §6.1 has not run")

## §7 Confirmation at n=96

n=48 puts a ±14-point interval on every rate, which is wide enough that most cells above will not
decide. `harmless_test` extends cleanly: it is `harmless[144 : 144 + test_n]`, so `test_n=96` keeps
the first 48 items **byte-identical** to every result this project has produced and adds 48 more.
§7.1 asserts that rather than trusting it.

Run this only after §4–§6 have decided what is worth confirming, and only with `ADASS_CONFIRM=1`:
it re-runs the closest pair — the best adaptive scheme against static at the same sparsity and
strength — at double n. Behavioural axes only; KL would need a second reference text and buys
nothing the rates do not.

In [ ]:
# %% 7.1 The closest pair at n=96.
if LOAD_MODEL and CONFIRM:
    SPL96 = adass.make_splits(seed=CONFIG["seed"], test_n=96)
    P96 = SPL96["harmless_test"]
    assert P96[:48] == PROMPTS, "the extended split moved the original 48 -- STOP, nothing is comparable"
    print(f"n={len(P96)}, first 48 identical to the standard split")

    H96 = adass.last_token_hidden(MODEL, TOK, TO_CHAT, P96, device=DEV)[LAYER + 1]
    G96 = adass.grad_scores(MODEL, TOK, TO_CHAT, P96, V, LAYER, device=DEV, dtype=DT, batch_size=4)

    def build96(method, sparsity):
        if method == "static":
            return (V * adass.static_mask(V, sparsity)).unsqueeze(0).expand(len(P96), -1).contiguous()
        out = torch.empty(len(P96), V.numel())
        for i in range(len(P96)):
            if method == "absproj":
                m = adass.adaptive_absproj_mask(V, H96[i], MU, sparsity)
            elif method == "signed":
                m = adass.adaptive_signed_mask(V, H96[i], MU, sparsity)
            elif method == "grad":
                m = adass.topk_mask(G96[i], sparsity)
            out[i] = V * m
        return out

    REL_C = REL_STAR or 1.0
    conf = {}
    for label, method in [("dense", None), ("static-0.90", "static"),
                          (f"{BEST_MASK}-0.90", BEST_MASK)]:
        base = V if method is None else build96(method, 0.90)
        row, gens, _ = run_config(label, base, REL_C, prompts=P96)
        conf[label] = dict(row=row, gens=gens)
        print(fmt(row, f"{label:14s} rel x{REL_C} (n=96)"))

    a, s = conf[f"{BEST_MASK}-0.90"]["row"], conf["static-0.90"]["row"]
    dec = disjoint(a["clean_refusal_ci"], s["clean_refusal_ci"])
    print(f"\n{BEST_MASK} vs static at n=96: clean {a['clean_refusal']:.1%} vs "
          f"{s['clean_refusal']:.1%} -> "
          + (("DECIDED, " + (BEST_MASK if a["clean_refusal"] > s["clean_refusal"] else "static"))
             if dec else "still undecided at n=96"))
    RESULTS["s7_confirmation"] = dict(cells=conf, rel_factor=REL_C, n=len(P96),
                                      decided=bool(dec), mask=BEST_MASK)
    print(adass.save_results(RESULTS, OUT))
else:
    print("skipped: needs ADASS_LOAD_MODEL=1 and ADASS_CONFIRM=1")

## §8 What this notebook settled

In [ ]:
# %% 8.1 Summary and the frontier figure.
import matplotlib.pyplot as plt

print("=" * 78)
for key, name in [("s1_gate", "replication gate"), ("s2_damage_onset", "damage onset"),
                  ("s4_h1_verdict", "H1"), ("s5_h2_verdict", "H2"), ("s6_h3_verdict", "H3"),
                  ("s7_confirmation", "n=96 confirmation")]:
    s = stored(key)
    if not s:
        print(f"{name:22s} not run")
    elif "verdict" in s:
        print(f"{name:22s} {s['verdict']}")
    elif key == "s2_damage_onset":
        print(f"{name:22s} REL_STAR = {s['rel_star']}")
    elif key == "s1_gate":
        print(f"{name:22s} {'PASS' if s['gate']['pass_'] else 'FAIL'}")
    else:
        print(f"{name:22s} decided={s.get('decided')}")
print("=" * 78)

s4, s2 = stored("s4_h1"), stored("s2_damage_onset")
if s4:
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.4))
    marks = {"dense": ("k", "o"), "static": ("tab:gray", "s"), "absproj": ("tab:blue", "^"),
             "signed": ("tab:green", "v"), "grad": ("tab:red", "D")}
    for key, c in s4["cells"].items():
        r = c["row"]
        col, mk = marks.get(r["method"], ("tab:purple", "x"))
        ax[0].scatter(r["suppressed"], 1 - r["broken"], c=col, marker=mk, s=70,
                      alpha=0.55 if r["rel_factor"] != 1.0 else 1.0,
                      edgecolors="k", linewidths=0.4)
        ax[1].scatter(r["suppressed"], r["kl"], c=col, marker=mk, s=70,
                      alpha=0.55 if r["rel_factor"] != 1.0 else 1.0,
                      edgecolors="k", linewidths=0.4)
    ax[0].set_xlabel("suppression (answering stopped)"); ax[0].set_ylabel("coherence (1 - broken)")
    ax[0].set_title("The frontier: effect against damage")
    ax[0].set_xlim(-0.05, 1.05); ax[0].set_ylim(-0.05, 1.05)
    ax[1].set_xlabel("suppression"); ax[1].set_ylabel("KL(steered || base)")
    ax[1].set_title("The damage axis with range left at rel 1.0")
    ax[1].set_xlim(-0.05, 1.05)
    for a in ax:
        a.grid(alpha=0.25)
    handles = [plt.Line2D([], [], color=c, marker=m, ls="", label=k) for k, (c, m) in marks.items()]
    ax[0].legend(handles=handles, fontsize=8, loc="lower left")
    fig.suptitle(f"AdaSS at layer {LAYER}: masking schemes at matched relative strength "
                 f"(solid = rel 1.0, faded = higher)", fontsize=10)
    plt.tight_layout()
    out_fig = adass.figure("fig_h1_frontier.png")
    plt.savefig(out_fig, dpi=150, bbox_inches="tight")
    print("figure ->", out_fig)
    plt.show()

RESULTS["meta"] = dict(notebook="07_h1_h3_layer10", layer=LAYER, r16=R16,
                       rel_points=REL_POINTS, max_new_tokens=MAXNEW,
                       strength="exact per-row norm matching (adass.rel_norm_rows), mult=1.0",
                       instruments="mechanical coherence + binary judge answered")
print(adass.save_results(RESULTS, OUT))